In [27]:
import random
import pandas as pd
from datetime import datetime, timedelta

In [ ]:
FIRST_NAMES = [
    "Alice", "Bob", "Carol", "David", "Eve", "Frank", "Grace", "Henry", "Ivy", "Jack",
    "Karen", "Leo", "Mia", "Noah", "Olivia", "Peter", "Quinn", "Rachel", "Sam", "Tina",
    "Uma", "Victor", "Wendy", "Xander", "Yara", "Zoe", "Aaron", "Bella", "Chris", "Diana",
    "Ethan", "Fiona", "George", "Hannah", "Ivan", "Julia", "Kevin", "Laura", "Mike", "Nina",
    "Oscar", "Paula", "Ryan", "Sara", "Tom", "Ursula", "Vince", "Willa", "Xena", "Yusuf",
    "Zara", "Adam", "Brooke", "Caleb", "Demi", "Eli", "Faith", "Gavin", "Holly", "Ian",
    "Jade", "Kyle", "Luna", "Mason", "Nora", "Owen", "Piper", "Reed", "Sofia", "Tyler",
    "Vera", "Wade", "Ximena", "Yasmine", "Zack", "Amber", "Blake", "Chloe", "Derek", "Elena",
    "Felix", "Gina", "Hugo", "Isla", "Jake", "Kylie", "Liam", "Maya", "Nathan", "Opal",
    "Patrick", "Rosa", "Scott", "Tara", "Uri", "Violet", "Will", "Xia", "York", "Zena",
]

LAST_NAMES = [
    "Johnson", "Smith", "Williams", "Brown", "Davis", "Miller", "Wilson", "Moore", "Taylor", "Anderson",
    "Thomas", "Jackson", "White", "Harris", "Martin", "Garcia", "Martinez", "Robinson", "Clark", "Lewis",
    "Lee", "Walker", "Hall", "Allen", "Young", "King", "Wright", "Scott", "Green", "Baker",
    "Adams", "Nelson", "Carter", "Mitchell", "Perez", "Roberts", "Turner", "Phillips", "Campbell", "Parker",
    "Evans", "Edwards", "Collins", "Stewart", "Sanchez", "Morris", "Rogers", "Reed", "Cook", "Morgan",
    "Bell", "Murphy", "Bailey", "Rivera", "Cooper", "Richardson", "Cox", "Howard", "Ward", "Torres",
    "Peterson", "Gray", "Ramirez", "James", "Watson", "Brooks", "Kelly", "Sanders", "Price", "Bennett",
    "Wood", "Barnes", "Ross", "Henderson", "Coleman", "Jenkins", "Perry", "Powell", "Long", "Patterson",
    "Hughes", "Flores", "Washington", "Butler", "Simmons", "Foster", "Gonzales", "Bryant", "Alexander", "Russell",
    "Griffin", "Diaz", "Hayes", "Myers", "Ford", "Hamilton", "Graham", "Sullivan", "Wallace", "Woods",
]

COUNTRIES   = ["US", "UK", "CA", "DE", "AU", "FR", "SG", "JP"]
DIAL_CODES  = {"US": "+1", "UK": "+44", "CA": "+1", "DE": "+49", "AU": "+61", "FR": "+33", "SG": "+65", "JP": "+81"}

def generate_raw_customers(n_customers=1000, seed=42, output_path="../seeds/raw_customers.csv"):
    random.seed(seed)
    records = []

    for i in range(1, n_customers + 1):
        first, last = random.choice(FIRST_NAMES), random.choice(LAST_NAMES)
        country = random.choice(COUNTRIES)
        digits  = random.randint(10_000_000, 9_999_999_999)
        date    = datetime(2026, 1, 1) + timedelta(days=random.randint(0, 365), hours=random.randint(7, 20))

        records.append({
            "customer_id":  f"cust_{i:04d}",
            "first_name":   first,
            "last_name":    last,
            "email":        f"{first.lower()}.{last.lower()}@email.com",
            "phone_number": f"{DIAL_CODES[country]} {digits}",
            "country":      country,
            "created_at":   date.strftime("%Y-%m-%d %H:%M:%S"),
        })

    df = pd.DataFrame(records)
    df.to_csv(output_path, index=False)
    return df

In [29]:
PRODUCTS = [
    ("Wireless Headphones", "Electronics", 49.99),
    ("Bluetooth Speaker",   "Electronics", 89.50),
    ("USB-C Hub",           "Electronics", 78.00),
    ("Mechanical Keyboard", "Electronics", 84.03),
    ("Webcam HD",           "Electronics", 89.00),
    ("Monitor Stand",       "Electronics", 124.50),
    ("Ergonomic Mouse",     "Electronics", 59.99),
    ("E-book Bundle",       "Digital Products", 19.99),
    ("Online Course Pack",  "Digital Products", 49.99),
    ("Music Stream Pass",   "Digital Products", 9.99),
    ("Desk Lamp",           "Lifestyle", 67.99),
    ("Yoga Mat",            "Lifestyle", 45.00),
    ("Water Bottle",        "Lifestyle", 24.99),
    ("Notebook Set",        "Lifestyle", 15.00),
    ("Desk Organizer",      "Lifestyle", 30.76),
    ("Running Shoes",       "Fashion", 94.00),
    ("Leather Wallet",      "Fashion", 39.99),
    ("Sunglasses",          "Fashion", 55.00),
    ("Shampoo Pack",        "FMCG", 12.50),
    ("Vitamin C Tablets",   "FMCG", 18.00),
]

def generate_raw_products(output_path="../seeds/raw_products.csv"):
    records = [
        {"product_id": f"prod_{i:03d}", "product_name": name, "category": cat, "price": price}
        for i, (name, cat, price) in enumerate(PRODUCTS, start=1)
    ]
    df = pd.DataFrame(records)
    df.to_csv(output_path, index=False)
    return df

In [30]:
def generate_raw_order_items(products_df, n_orders=500, seed=42, output_path="../seeds/raw_order_items.csv"):
    random.seed(seed)
    # Use product prices from products_df instead of random prices
    price_map = dict(zip(products_df["product_id"], products_df["price"]))
    product_ids = products_df["product_id"].tolist()
    records = []
    item_counter = 1

    for order_num in range(1, n_orders + 1):
        for _ in range(random.randint(1, 4)):
            product_id = random.choice(product_ids)
            records.append({
                "order_item_id": f"oi_{item_counter:03d}",
                "order_id":      f"ord_{order_num:03d}",
                "product_id":    product_id,
                "quantity":      random.randint(1, 5),
                "unit_price":    price_map[product_id],
            })
            item_counter += 1

    df = pd.DataFrame(records)
    df.to_csv(output_path, index=False)
    return df

In [31]:
def generate_raw_orders(order_items_df, n_customers=1000, seed=42, output_path="../seeds/raw_orders.csv"):
    random.seed(seed)
    STATUSES = ["delivered", "shipped", "processing", "returned", "cancelled"]

    # Calculate total_amount per order from order items
    order_totals = (
        order_items_df
        .assign(line_total=lambda df: df["quantity"] * df["unit_price"])
        .groupby("order_id")["line_total"]
        .sum()
        .round(2)
    )

    records = []
    for i, (order_id, total) in enumerate(order_totals.items(), start=1):
        date = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 365))
        records.append({
            "order_id":     order_id,
            "customer_id":  f"cust_{random.randint(1, n_customers):04d}",
            "order_date":   date.strftime("%Y-%m-%d"),
            "status":       random.choice(STATUSES),
            "total_amount": total,
        })

    df = pd.DataFrame(records)
    df.to_csv(output_path, index=False)
    return df


In [39]:
n_customers = 1000
n_orders = 500

df_customers   = generate_raw_customers(n_customers=n_customers)
df_products    = generate_raw_products()
df_order_items = generate_raw_order_items(products_df=df_products , n_orders = n_orders)
df_orders      = generate_raw_orders(order_items_df=df_order_items, n_customers = n_customers)

In [40]:
df_customers

,customer_id,first_name,last_name,email,phone_number,country,created_at
0,cust_0001,Gina,Martin,gina.martin@email.com,+1 7489902459,US,6/5/26 10:00
1,cust_0002,Rachel,Ford,rachel.ford@email.com,+44 8973334018,UK,5/8/26 07:00
2,cust_0003,David,Jackson,david.jackson@email.com,+49 9599205528,DE,5/11/26 07:00
3,cust_0004,Wade,King,wade.king@email.com,+65 5251752544,SG,29/10/26 11:00
4,cust_0005,Alice,Sullivan,alice.sullivan@email.com,+1 7303453178,CA,24/6/26 11:00
...,...,...,...,...,...,...,...
995,cust_0996,Gavin,Price,gavin.price@email.com,+61 6787163837,AU,11/4/26 10:00
996,cust_0997,Maya,Evans,maya.evans@email.com,+61 8983421080,AU,5/1/26 16:00
997,cust_0998,Owen,Murphy,owen.murphy@email.com,+1 7966449152,CA,1/2/26 12:00
998,cust_0999,Vera,Wilson,vera.wilson@email.com,+1 1552943951,US,25/5/26 20:00
